# System cost decomposition by country

Decomposes the **total system cost** of a solved PyPSA-DE network, step by step,
using **only the cost-accounting functions that the workflow itself uses**
(`get_tsc`, `get_trade_cost`, `get_export_import` from `export_ariadne_variables`;
`total_cost_stats`, `de_total_cost`, `outside_de_non_trade_cost`, `eu_level_cost`
from `compute_stochastic_metrics`). All numbers are shown in **bn EUR**.

Order:
1. the LP **objective** and what feeds it,
2. the **total** system cost via `n.statistics` (capex + opex),
3. split into **DE** and **outside-DE**,
4. split into **all countries** + the shared **EU-level** buses (sums to the total),
5. the **capex / opex / trade** split (interconnectors shared 50/50), and
6. a **sum check across several runs**, LT and ST.

In [1]:
import os
import sys

ROOT = "/home/julian-geis/repos/pypsa-de"
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "scripts", "pypsa-de"))
os.chdir(ROOT)

import pandas as pd
import pypsa
from IPython.display import display

# cost accounting reused verbatim from the workflow
from export_ariadne_variables import get_tsc, get_trade_cost, get_export_import
from compute_stochastic_metrics import (
    total_cost_stats,
    de_total_cost,
    outside_de_non_trade_cost,
    de_trade_cost,
    TRADE_CARRIER_GROUPS,
)

REGION = 'DE'
bn = lambda x: round(float(x) / 1e9, 2)  # EUR -> bn EUR

## Example networks
Three different runs, each a long-term topology (**LT**) and a short-term dispatch evaluation (**ST**).

In [2]:
NETWORKS = {
    "run1 LT (per-scenario-kvl, Mix)": "results/20261006-per-scenario-kvl-49cl-365h-2025EXO-2030EXO-2035EXO-10-50-40/KN2045_Mix/networks/base_s_49__none_2035_topology-2035_exogen.nc",
    "run1 ST (per-scenario-kvl, Mix)": "results/20261006-per-scenario-kvl-49cl-365h-2025EXO-2030EXO-2035EXO-10-50-40/KN2045_Mix/networks/base_s_49__none_2035_portfolio-2025_exogen_on-2035_exogen_st.nc",
    "run2 LT (Bal_v5)": "results/20261014-KN2045_Bal_v5-49cl-365h-2025EXO-2030EXO-2035EXO-10-50-40/KN2045_Bal_v5/networks/base_s_49__none_2035_topology-2035_exogen.nc",
    "run2 ST (Bal_v5)": "results/20261014-KN2045_Bal_v5-49cl-365h-2025EXO-2030EXO-2035EXO-10-50-40/KN2045_Bal_v5/networks/base_s_49__none_2035_portfolio-2025_exogen_on-2035_exogen_st.nc",
    "run3 LT (test, Mix)": "results/20260930_test-49cl-365h-2025EXO-2030EXO-2035EXO-10-30-60/KN2045_Mix/networks/base_s_49__none_2035_topology-2035_exogen.nc",
    "run3 ST (test, Mix)": "results/20260930_test-49cl-365h-2025EXO-2030EXO-2035EXO-10-30-60/KN2045_Mix/networks/base_s_49__none_2035_portfolio-2025_exogen_on-2035_exogen_st.nc",
}

EXAMPLE = "run1 LT (per-scenario-kvl, Mix)"
n = pypsa.Network(NETWORKS[EXAMPLE])
EXAMPLE

INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks


'run1 LT (per-scenario-kvl, Mix)'

## 1. The LP objective
`n.objective + n.objective_constant` is what the solver minimises. It is the cost of
the **decisions**: operation (opex) plus **new** investment (expanded capex). It does
**not** include the capex of already-built (non-extendable) capacity - that sunk cost
sits in `objective_constant` (often 0 here). So the objective is **not** the full
annualised system cost; section 2 is.

In [3]:
print("objective            ", bn(n.objective), "bn")
print("objective_constant   ", bn(n.objective_constant), "bn")
print("objective + constant ", bn(n.objective + n.objective_constant), "bn")

opex = n.statistics.opex().sum()
expanded = n.statistics.expanded_capex().sum()  # new investment only
print()
print("opex (operation)             ", bn(opex), "bn")
print("expanded_capex (new invest.) ", bn(expanded), "bn")
print("opex + expanded_capex        ", bn(opex + expanded), "bn  (~ objective)")

objective             515.65 bn
objective_constant    0.0 bn
objective + constant  515.65 bn

opex (operation)              189.33 bn
expanded_capex (new invest.)  326.32 bn
opex + expanded_capex         515.65 bn  (~ objective)


## 2. Total system cost via `n.statistics`
The full annualised system cost = capex of **all** assets (incl. already-built capacity
and the grid) + opex. This is `total_cost_stats`, and the basis the DE/country costs use.

In [4]:
capex_all = n.statistics.capex().sum()
total = total_cost_stats(n)
print("statistics capex (ALL) ", bn(capex_all), "bn")
print("statistics opex        ", bn(opex), "bn")
print("TOTAL system cost      ", bn(total), "bn")

statistics capex (ALL)  723.99 bn
statistics opex         189.33 bn
TOTAL system cost       913.32 bn


## 3. Split into DE and outside-DE
`get_tsc(DE)` is the DE share (capex+opex, interconnectors split 50/50).
`outside_de_non_trade_cost` is its exact complement (whole-network total minus the
DE share): every other country plus the shared EU-level buses. They reconstruct the
total exactly.

In [5]:
de = get_tsc(n, REGION).values.sum()
outside = outside_de_non_trade_cost(n, REGION)
print("DE non-trade       ", bn(de), "bn  (", round(100 * de / total), "% )")
print("outside-DE non-trade", bn(outside), "bn")
print("DE + outside       ", bn(de + outside), "bn   vs total", bn(total), "bn")
print("residual           ", round(100 * abs(total - (de + outside)) / total, 2), "%")

DE non-trade        171.05 bn  ( 19 % )
outside-DE non-trade 742.27 bn
DE + outside        913.32 bn   vs total 913.32 bn
residual            0.0 %


## 4. Split into all countries + EU-level
Each country via `get_tsc` (capex and opex, interconnectors split 50/50). The shared
primary-energy buses (`EU oil/gas/uranium/coal`) carry no country code, so `get_tsc`
attributes them to no country; they are the **EU-level** remainder = statistics total
minus the sum over countries. The rows sum back to the total **exactly**.

In [6]:
countries = sorted(c for c in n.buses.country.unique() if c and c != 'EU')
rows = {c: {'capex': get_tsc(n, c)['capex'].sum(), 'opex': get_tsc(n, c)['opex'].sum()} for c in countries}
tab = pd.DataFrame(rows).T

# EU-level = the exact remainder get_tsc attributes to no country
tab.loc['EU-level'] = {
    'capex': n.statistics.capex().sum() - tab['capex'].sum(),
    'opex': n.statistics.opex().sum() - tab['opex'].sum(),
}

tab['capex+opex'] = tab['capex'] + tab['opex']
display((tab / 1e9).round(2).sort_values('capex+opex', ascending=False))

print("sum over all rows  ", bn(tab["capex+opex"].sum()), "bn   vs total", bn(total), "bn")
print("residual           ", round(100 * abs(total - tab["capex+opex"].sum()) / total, 4), "%")

,capex,opex,capex+opex
DE,132.98,38.07,171.05
FR,132.71,6.96,139.67
IT,109.38,6.51,115.89
EU-level,-10.41,115.96,105.55
GB,99.62,3.23,102.86
ES,55.92,3.04,58.96
PL,42.94,4.01,46.94
NL,29.22,2.12,31.34
CH,26.95,2.94,29.89
SE,24.02,1.60,25.62


sum over all rows   913.32 bn   vs total 913.32 bn
residual            0.0 %


## 5. capex / opex / trade
Non-trade (capex+opex) is attributed above. **Trade** is a transfer: DE pays imports and
earns exports, valued at domestic marginal prices (`get_export_import`, `get_trade_cost`).
It is *not* part of the whole-system capex+opex total: summed over all countries the
net trade does not vanish but equals the **interconnector congestion rent** (imports are
valued at the importer's price, exports at the exporter's - the gap is the line's rent).

In [7]:
# DE trade: import cost vs export revenue per traded carrier group
trade_rows = {}
for grp in TRADE_CARRIER_GROUPS:
    rev, cost = get_export_import(n, REGION, grp, unit='€')
    trade_rows[' + '.join(grp)] = {'import_cost': cost, 'export_revenue': rev, 'net (cost-rev)': cost - rev}
trade = pd.DataFrame(trade_rows).T
display((trade / 1e9).round(3))
print("DE net trade (de_trade_cost)        ", bn(de_trade_cost(n, REGION)), "bn")

# DE as capex / opex / trade
tsc_de = get_tsc(n, REGION)
print()
print("DE capex    ", bn(tsc_de["capex"].sum()), "bn")
print("DE opex     ", bn(tsc_de["opex"].sum()), "bn")
print("DE net trade", bn(de_trade_cost(n, REGION)), "bn")

# trade nets out across the whole system
net_trade_all = sum(
    sum(get_trade_cost(n, c, grp) for grp in TRADE_CARRIER_GROUPS) for c in countries
)
print()
print("sum of net trade over ALL countries", bn(net_trade_all), "bn  (= interconnector congestion rent)")

,import_cost,export_revenue,net (cost-rev)
AC,8.891,4.685,4.207
DC,2.195,0.688,1.507
renewable oil + renewable gas + methanol,0.432,0.004,0.427
H2 pipeline + H2 pipeline (Kernnetz) + H2 pipeline retrofitted,9.288,4.602,4.686


DE net trade (de_trade_cost)         10.83 bn



DE capex     132.98 bn
DE opex      38.07 bn
DE net trade 10.83 bn



sum of net trade over ALL countries 15.4 bn  (= interconnector congestion rent)


## 6. Sum check across runs (LT and ST)
`total == DE + outside-DE` for every example network - exact, since outside-DE is the
complement of the DE share.

In [8]:
tol = 0.03
res = {}
for name, path in NETWORKS.items():
    m = pypsa.Network(path)
    tot = total_cost_stats(m)
    d = get_tsc(m, REGION).values.sum()
    o = outside_de_non_trade_cost(m, REGION)
    res[name] = {
        'total': tot / 1e9,
        'DE': d / 1e9,
        'outside-DE': o / 1e9,
        'DE+outside': (d + o) / 1e9,
        'residual_%': 100 * abs(tot - (d + o)) / tot,
    }
check = pd.DataFrame(res).T.round(2)
display(check)
assert (check['residual_%'] < 100 * tol).all(), 'decomposition residual exceeds tolerance'
print(f'All {len(NETWORKS)} networks decompose within {100 * tol:.0f}% ✓')

INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.4.0 available! (Current:

,total,DE,outside-DE,DE+outside,residual_%
"run1 LT (per-scenario-kvl, Mix)",913.32,171.05,742.27,913.32,0.0
"run1 ST (per-scenario-kvl, Mix)",915.40,173.10,742.30,915.40,0.0
run2 LT (Bal_v5),766.86,150.84,616.02,766.86,0.0
run2 ST (Bal_v5),768.51,153.87,614.64,768.51,0.0
"run3 LT (test, Mix)",912.55,169.78,742.77,912.55,0.0
"run3 ST (test, Mix)",914.61,171.81,742.80,914.61,0.0


All 6 networks decompose within 3% ✓


## 7. Per-country cost *including* trade (full reconciliation)
Joins the per-country non-trade cost (section 4) with its net trade (section 5) into
one `incl_trade` column: each country's own assets **plus its net import bill** (net
importers go up, net exporters down). Trade is a transfer, so summed over all countries
it does not vanish - it equals the interconnector **congestion rent** (prior section),
which the model books to no country. The statistics total is therefore recovered only
with two explicit rows: the shared **EU-level** buses added, and the **rent** subtracted:

`total == sum(incl_trade over countries) + EU-level - rent`, exactly.

In [9]:
# per-country non-trade (get_tsc) + net trade, reconciled to the statistics total
rows = {}
for c in countries:
    non_trade = get_tsc(n, c).values.sum()
    trade = sum(get_trade_cost(n, c, grp) for grp in TRADE_CARRIER_GROUPS)
    rows[c] = {'non_trade': non_trade, 'trade': trade, 'incl_trade': non_trade + trade}
tab = pd.DataFrame(rows).T

eu_level = total - tab['non_trade'].sum()  # shared buses get_tsc attributes to no country
rent = tab['trade'].sum()                  # sum of net trade over all countries = congestion rent

display((tab.sort_values('incl_trade', ascending=False) / 1e9).round(2))

reconciled = tab['incl_trade'].sum() + eu_level - rent
print("sum incl_trade over countries", bn(tab['incl_trade'].sum()), "bn")
print("+ EU-level                   ", bn(eu_level), "bn")
print("- congestion rent            ", bn(rent), "bn")
print("= reconciled total           ", bn(reconciled), "bn   vs total", bn(total), "bn")
print("residual                     ", round(100 * abs(total - reconciled) / total, 4), "%")
assert abs(reconciled - total) / total < 1e-6

,non_trade,trade,incl_trade
DE,171.05,10.83,181.88
FR,139.67,4.52,144.18
IT,115.89,-1.09,114.80
GB,102.86,-8.66,94.20
ES,58.96,0.08,59.04
PL,46.94,2.53,49.47
NL,31.34,1.82,33.15
CH,29.89,-1.31,28.57
SE,25.62,0.97,26.59
AT,19.63,2.51,22.13


sum incl_trade over countries 823.17 bn
+ EU-level                    105.55 bn
- congestion rent             15.4 bn
= reconciled total            913.32 bn   vs total 913.32 bn
residual                      0.0 %
